# 2. Transpose
* This notebook demonstrates how transpose operations work on tensors
* For 2D tensors, transpose swaps positions from (x, y) to (y, x), but what about multi-dimensional tensors?
* For multi-dimensional tensors, transpose reorders the dimensions (axes). For example:
  * Given a tensor with shape (x, y, z), applying `transpose(0, 2, 1)` results in shape (x, z, y)
  * Applying `transpose(1, 2, 0)` results in shape (y, z, x)
* The new tensor's dimensions follow the order specified in the permutation
* We'll explore three implementations: naive element-by-element, memory-efficient view-based, and PyTorch's built-in method

## Method 1: Naive Implementation (Element-by-Element Copy)

This implementation explicitly copies each element to its new position. It is done by visiting the elements.

In [27]:
import torch
def transpose_simple(input, perm):
    """
    input: torch.Tensor of any dimension
    perm: tuple/list specifying new axis order
    """
    assert len(input.shape) == len(perm)

    # output shape after permutation
    out_shape = [input.shape[p] for p in perm]
    out = torch.empty(out_shape, dtype=input.dtype)

    ndim = input.dim()

    # index for input tensor
    idx = [0] * ndim

    while True:
        # compute output index by permuting input index
        out_idx = [idx[p] for p in perm]
        out[tuple(out_idx)] = input[tuple(idx)]

        # increment idx (like an N-dimensional counter)
        for d in reversed(range(ndim)):
            idx[d] += 1
            if idx[d] < input.shape[d]:
                break
            idx[d] = 0
        else:
            # fully overflowed -> done
            break

    return out

In [28]:
# Create a test tensor
torch.manual_seed(42)
test_tensor = torch.rand(2, 3, 4)

print("Original Tensor:")
print(f"Shape: {test_tensor.shape}")
print(f"Content:\n{test_tensor}")

# Permutation: swap last two dimensions (0, 2, 1)
perm = (0, 2, 1)

transposed_simple = transpose_simple(test_tensor, perm)

print("Transposed Tensor:")
print(f"Shape: {transposed_simple.shape}")
print(f"Content:\n{transposed_simple}")

Original Tensor:
Shape: torch.Size([2, 3, 4])
Content:
tensor([[[0.8823, 0.9150, 0.3829, 0.9593],
         [0.3904, 0.6009, 0.2566, 0.7936],
         [0.9408, 0.1332, 0.9346, 0.5936]],

        [[0.8694, 0.5677, 0.7411, 0.4294],
         [0.8854, 0.5739, 0.2666, 0.6274],
         [0.2696, 0.4414, 0.2969, 0.8317]]])
Transposed Tensor:
Shape: torch.Size([2, 4, 3])
Content:
tensor([[[0.8823, 0.3904, 0.9408],
         [0.9150, 0.6009, 0.1332],
         [0.3829, 0.2566, 0.9346],
         [0.9593, 0.7936, 0.5936]],

        [[0.8694, 0.8854, 0.2696],
         [0.5677, 0.5739, 0.4414],
         [0.7411, 0.2666, 0.2969],
         [0.4294, 0.6274, 0.8317]]])


## Method 2: View-Based
This implementation uses PyTorch's stride mechanism to create a view of the original tensor without copying data. It shares data with the original version.

In [29]:
def transpose_view(input, perm):
    """
    Returns a VIEW of `input` with dimensions permuted.
    No data is copied.

    input: torch.Tensor
    perm: tuple/list specifying new axis order
    """
    assert len(input.shape) == len(perm)

    # New shape is just a reordering of old shape
    new_shape = [input.shape[p] for p in perm]

    # New strides are reordered the same way
    new_stride = [input.stride()[p] for p in perm]

    # as_strided creates a view using shape + stride metadata
    return torch.as_strided(input, size=new_shape, stride=new_stride)

In [30]:

# Use the same test tensor for consistency
torch.manual_seed(42)
test_tensor_view = torch.rand(2, 3, 4)

print("Original Tensor:")
print(f"Shape: {test_tensor_view.shape}")
print(f"Content:\n{test_tensor_view}")

# Use the same permutation
perm = (0, 2, 1)

transposed_view = transpose_view(test_tensor_view, perm)

print("\nTransposed Tensor (Method 2 - View):")
print(f"Shape: {transposed_view.shape}")
print(f"Content:\n{transposed_view}")

# Demonstrate that this is a view (shares memory)
print("\n--- Demonstrating View Behavior ---")
print(f"Shares memory with original: {transposed_view.data_ptr() == test_tensor_view.data_ptr()}")
print(f"Is contiguous: {transposed_view.is_contiguous()}")

# Modify the original tensor
test_tensor_view[0, 0, 0] = 999.0
print("\nAfter setting original[0,0,0] = 999.0:")
print(f"Original[0,0,0] = {test_tensor_view[0,0,0]}")
print(f"Transposed view[0,0,0] = {transposed_view[0,0,0]} (also changed!)")

Original Tensor:
Shape: torch.Size([2, 3, 4])
Content:
tensor([[[0.8823, 0.9150, 0.3829, 0.9593],
         [0.3904, 0.6009, 0.2566, 0.7936],
         [0.9408, 0.1332, 0.9346, 0.5936]],

        [[0.8694, 0.5677, 0.7411, 0.4294],
         [0.8854, 0.5739, 0.2666, 0.6274],
         [0.2696, 0.4414, 0.2969, 0.8317]]])

Transposed Tensor (Method 2 - View):
Shape: torch.Size([2, 4, 3])
Content:
tensor([[[0.8823, 0.3904, 0.9408],
         [0.9150, 0.6009, 0.1332],
         [0.3829, 0.2566, 0.9346],
         [0.9593, 0.7936, 0.5936]],

        [[0.8694, 0.8854, 0.2696],
         [0.5677, 0.5739, 0.4414],
         [0.7411, 0.2666, 0.2969],
         [0.4294, 0.6274, 0.8317]]])

--- Demonstrating View Behavior ---
Shares memory with original: True
Is contiguous: False

After setting original[0,0,0] = 999.0:
Original[0,0,0] = 999.0
Transposed view[0,0,0] = 999.0 (also changed!)


## Method 3: PyTorch's Built-in Permute

In [31]:
# Finally what you should normaly use
def transpose_permute(input, perm):
    return input.permute(*perm)

In [32]:
# Use the same test tensor for consistency
torch.manual_seed(42)
test_tensor_permute = torch.rand(2, 3, 4)

print("Original Tensor:")
print(f"Shape: {test_tensor_permute.shape}")
print(f"Content:\n{test_tensor_permute}")

# Use the same permutation
perm = (0, 2, 1)

transposed_permute = transpose_permute(test_tensor_permute, perm)

print("\nTransposed Tensor (Method 3 - Permute):")
print(f"Shape: {transposed_permute.shape}")
print(f"Content:\n{transposed_permute}")

# Verify this is also a view
print(f"\nShares memory with original: {transposed_permute.data_ptr() == test_tensor_permute.data_ptr()}")

Original Tensor:
Shape: torch.Size([2, 3, 4])
Content:
tensor([[[0.8823, 0.9150, 0.3829, 0.9593],
         [0.3904, 0.6009, 0.2566, 0.7936],
         [0.9408, 0.1332, 0.9346, 0.5936]],

        [[0.8694, 0.5677, 0.7411, 0.4294],
         [0.8854, 0.5739, 0.2666, 0.6274],
         [0.2696, 0.4414, 0.2969, 0.8317]]])

Transposed Tensor (Method 3 - Permute):
Shape: torch.Size([2, 4, 3])
Content:
tensor([[[0.8823, 0.3904, 0.9408],
         [0.9150, 0.6009, 0.1332],
         [0.3829, 0.2566, 0.9346],
         [0.9593, 0.7936, 0.5936]],

        [[0.8694, 0.8854, 0.2696],
         [0.5677, 0.5739, 0.4414],
         [0.7411, 0.2666, 0.2969],
         [0.4294, 0.6274, 0.8317]]])

Shares memory with original: True
